# 05 · Build business metrics with CTEs

Create an auditable daily operating report with explicit quality filters and exact money totals.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
from decimal import Decimal
taxi = lab.taxi()
config = lab.catalog(sources=[{"id": "trips", "type": "parquet", "path": str(taxi)}])
valid_filter = """
    tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
    AND tpep_pickup_datetime < TIMESTAMP '2024-02-01'
    AND trip_distance > 0 AND total_amount >= 0 AND total_amount < 1000
"""


## Separate selection, normalization and reporting

These are trip-level indicators, not a representation of any ride-hailing
company's internal accounting. `total_amount` follows the TLC field definition;
it is not automatically platform revenue or profit.


In [ ]:
daily = lab.query(config, f"""
    WITH eligible AS (
        SELECT tpep_pickup_datetime, trip_distance, payment_type,
               total_amount::DECIMAL(18,2) AS collected_amount
        FROM trips WHERE {valid_filter}
    ), daily_totals AS (
        SELECT CAST(tpep_pickup_datetime AS DATE) AS day,
               COUNT(*)::BIGINT AS trips,
               SUM(collected_amount) AS collected_amount,
               COUNT(*) FILTER (WHERE payment_type = 1)::BIGINT AS card_trips,
               AVG(trip_distance) AS mean_distance
        FROM eligible GROUP BY 1
    )
    SELECT *, collected_amount / NULLIF(trips, 0) AS collected_per_trip,
           100.0 * card_trips / NULLIF(trips, 0) AS card_share_pct
    FROM daily_totals ORDER BY day
""", sources=["trips"], max_rows=40).table()
display(daily.to_pandas())


## Reconcile the report to its input population

The money check uses Python `Decimal`; it does not round a floating-point total
back into apparent agreement.


In [ ]:
control = lab.query(config, f"""
    SELECT COUNT(*)::BIGINT AS trips,
           SUM(total_amount::DECIMAL(18,2)) AS collected_amount
    FROM trips WHERE {valid_filter}
""", sources=["trips"]).table().to_pylist()[0]
assert 1 <= daily.num_rows <= 31
assert sum(daily["trips"].to_pylist()) == control["trips"]
assert sum(daily["collected_amount"].to_pylist(), Decimal(0)) == control["collected_amount"]
assert all(0 <= value <= 100 for value in daily["card_share_pct"].to_pylist())
print("Daily report reconciles:", control)


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
